# Heart Disease: EDA, modelling and inference

Run from the repository root after `python scripts/download_data.py`. The same logic lives in `src/heart/` and `scripts/`; this notebook is the interactive walkthrough.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from heart import config
from heart.data import load_clean, split_xy
sns.set_theme(style='whitegrid')
df = load_clean('../data/processed/heart_clean.csv')
df.head()

## 1. Data overview

In [ ]:
print(df.shape)
display(df.describe().T.round(2))
print(df.isna().sum()[df.isna().sum() > 0])

## 2. Class balance and distributions

In [ ]:
df[config.TARGET].value_counts().plot.bar(color=['#4c78a8', '#e45756'], rot=0, title='Class balance');

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), config.NUMERIC_FEATURES):
    sns.histplot(df, x=c, hue=config.TARGET, kde=True, ax=ax, palette={0: '#4c78a8', 1: '#e45756'})
plt.tight_layout()

## 3. Correlations

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation heatmap');

## 4. Pipeline and cross-validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from heart.features import build_pipeline
X_train, X_test, y_train, y_test = split_xy(df)
cv = StratifiedKFold(5, shuffle=True, random_state=config.RANDOM_STATE)
rows = []
for name, est in {'logreg': LogisticRegression(max_iter=2000),
                  'random_forest': RandomForestClassifier(300, random_state=42)}.items():
    res = cross_validate(build_pipeline(est), X_train, y_train, cv=cv,
                         scoring=['accuracy', 'precision', 'recall', 'roc_auc'])
    rows.append({'model': name, **{m: res[f'test_{m}'].mean() for m in ['accuracy', 'precision', 'recall', 'roc_auc']}})
pd.DataFrame(rows).round(3)

Full tuning (GridSearchCV), MLflow logging and model selection: `python -m heart.train`.

## 5. Inference with the saved model

In [ ]:
from heart.predict import load_model, predict_records
model = load_model('../models/model.joblib')
patient = {'age': 63, 'sex': 1, 'cp': 4, 'trestbps': 145, 'chol': 233, 'fbs': 1, 'restecg': 2,
           'thalach': 150, 'exang': 0, 'oldpeak': 2.3, 'slope': 3, 'ca': 0, 'thal': 6}
predict_records(model, [patient])